# 1. ProteinMPNN: new sequences for a known structure
**Question:** How much can a sequence change while remaining compatible with the same backbone?

We keep the 76-residue ubiquitin backbone (1UBQ, chain A) fixed and ask ProteinMPNN for three sequences. Then we compare their sequences and predict one structure. **Target: under 10 minutes after setup on a free Colab GPU; timing needs a classroom trial.**


## Before class
Choose **Runtime → Change runtime type → T4 GPU**, with **runtime version 2026.07** (already selected in this notebook). Run the two setup cells once and keep the runtime connected. The shared tools can be reused later in the class. ESMFold needs a large download; setup time is separate from this demo. Start the timed walkthrough at step 1.


In [ ]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


In [ ]:
load_MPNN()
load_ESMFold()


## 1. Start with a structure
Rotate ubiquitin and find its helices, sheets, and loops. This is the structural constraint for sequence design.


In [ ]:
native_seq, reference_pdb = load_pdb('1UBQ', chain='A')
view_structure(reference_pdb)


## 2. Design sequences
ProteinMPNN samples amino acids conditioned on the backbone. Higher temperature usually produces more diverse sequences. We do not require recovery of the natural sequence.


In [ ]:
designs = MPNNdesign(reference_pdb, num_sequences=3, temperature=0.1, seed=7)


## 3. Compare sequences
`|` marks a match to ubiquitin; `.` marks a substitution. Which positions are conserved across all three designs?


In [ ]:
alignment({'ubiquitin':native_seq, **designs})


## 4. Does the sequence predict the intended fold?
Fold just one sequence to keep the demo short. ESMFold predicts from sequence alone; it does not receive our reference coordinates. One recycle is a fast check. RMSD measures agreement with the reference; pLDDT measures the predictor's local confidence. Neither demonstrates experimental folding or function.


In [ ]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


## Inspect
- Where does the prediction agree with the backbone, and where does it differ? Check confidence at the mismatches.
- Would a matching fold establish that the design retains ubiquitin's biological function? What else would you test?

[ProteinMPNN](https://github.com/dauparas/ProteinMPNN) · [1UBQ](https://www.rcsb.org/structure/1UBQ) · [ESMFold](https://github.com/facebookresearch/esm)


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))
